In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [7]:
df = pd.read_csv('spamhamdata.csv',sep='\t',names=['label', 'message'])

In [8]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
X = df['message']
y = df['label']

In [10]:
y = y.map({'ham':0, 'spam':1})

In [11]:
y.value_counts()

label
0    4825
1     747
Name: count, dtype: int64

In [12]:
import nltk
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [13]:
lemmatizer = WordNetLemmatizer()

In [14]:
corpus = []
for i in range(len(X)):
    review = re.sub('[^a-zA-Z]', ' ', X[i])
    review = review.lower()
    review = review.split()
    review = [lemmatizer.lemmatize(word) for word in review if not word in set(stopwords.words('english'))]
    review = ' '.join(review)
    corpus.append(review)

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(corpus, y, test_size=0.2, random_state=0)

In [17]:
tfidf = TfidfVectorizer(max_features=2500, ngram_range=(1,3))

In [18]:
X_train = tfidf.fit_transform(X_train).toarray()
X_test = tfidf.transform(X_test).toarray()

In [19]:
tfidf.vocabulary_

{'good': np.int64(817),
 'movie': np.int64(1355),
 'ok': np.int64(1462),
 'leave': np.int64(1104),
 'free': np.int64(709),
 'give': np.int64(797),
 'otherwise': np.int64(1497),
 'something': np.int64(1893),
 'maybe': np.int64(1254),
 'bit': np.int64(160),
 'got': np.int64(831),
 'home': np.int64(941),
 'babe': np.int64(118),
 'still': np.int64(1950),
 'awake': np.int64(110),
 'since': np.int64(1863),
 'already': np.int64(45),
 'workin': np.int64(2438),
 'get': np.int64(775),
 'job': np.int64(1041),
 'said': np.int64(1769),
 'matter': np.int64(1247),
 'mind': np.int64(1284),
 'saying': np.int64(1789),
 'oh': np.int64(1459),
 'yeah': np.int64(2480),
 'oh yeah': np.int64(1461),
 'sorry': np.int64(1900),
 'thing': np.int64(2090),
 'may': np.int64(1252),
 'pub': np.int64(1654),
 'later': np.int64(1087),
 'ill': np.int64(986),
 'call': np.int64(215),
 'evening': np.int64(606),
 'idea': np.int64(979),
 'dear': np.int64(494),
 'room': np.int64(1750),
 'look': np.int64(1157),
 'daddy': np.int64

In [20]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report

In [21]:
models = {
    'Naive Bayes': MultinomialNB(),
    'Logistic Regression': LogisticRegression(),
    'SVM': SVC(kernel='linear'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=0)
}

In [22]:
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f'{name} Accuracy: {accuracy_score(y_test, y_pred)}')
    print(f'{name} Classification Report:\n{classification_report(y_test, y_pred)}')
    print(f'{name} Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}\n')

Naive Bayes Accuracy: 0.9766816143497757
Naive Bayes Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       955
           1       0.99      0.84      0.91       160

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115

Naive Bayes Confusion Matrix:
[[954   1]
 [ 25 135]]

Logistic Regression Accuracy: 0.9730941704035875
Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       955
           1       0.99      0.82      0.90       160

    accuracy                           0.97      1115
   macro avg       0.98      0.91      0.94      1115
weighted avg       0.97      0.97      0.97      1115

Logistic Regression Confusion Matrix:
[[954   1]
 [ 29 131]]

SVM Accuracy: 0.9865470852017937
SVM Classification Report:
  

#### SVM is performing best out of all

In [29]:
from sklearn.model_selection import GridSearchCV,RandomizedSearchCV

In [24]:
## hyperparameter tuning for SVM
svm_params = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf'],
}

In [ ]:
grid = GridSearchCV(SVC(probability=True), svm_params, cv=3, n_jobs=-1,scoring='accuracy')

In [30]:
random_cv = RandomizedSearchCV(SVC(probability=True), svm_params, cv=3, n_jobs=-1,scoring='accuracy', n_iter=5, random_state=0)

In [31]:
random_cv.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",SVC(probability=True)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'C': [0.1, 1, ...], 'kernel': ['linear', 'rbf']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",5
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"v

In [32]:
random_cv.best_params_

{'kernel': 'linear', 'C': 1}

In [35]:
import pickle
import joblib
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(random_cv, 'svm_model.pkl') 

['svm_model.pkl']

In [38]:
def make_prediction(text):
    model = joblib.load('svm_model.pkl')
    vectorizer = joblib.load('tfidf_vectorizer.pkl')
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text if not word in set(stopwords.words('english'))]
    text = ' '.join(text)
    text = vectorizer.transform([text]).toarray()
    prediction = model.predict(text)
    return prediction


In [39]:
print(make_prediction("Congratulations! You've won a free ticket to the Bahamas. Click here to claim now!"))

[1]


In [40]:
## ham input
print(make_prediction("Hey, are we still on for dinner tonight?"))

[0]


In [41]:
## trick input
print(make_prediction("Hey, are we still on for dinner tonight? Click here to claim your free ticket to the Bahamas!"))

[0]


In [42]:
## some more testing
print(make_prediction("Don't miss out on this amazing offer! Get your free ticket to the Bahamas now!"))

[0]


In [43]:
print(make_prediction("Congratulations! You've won a free ticket to the Bahamas. Click here to claim now!"))

[1]
